# Tutorial 14: End-to-End Coherent Scattering Experiment

This is the maintained replacement for the old monolithic scattering notebooks. It defines one complete magnetic Fourier-transform holography experiment, simulates matched CR and CL measurements, and compares charge-like sums with magnetic helicity differences at every stage.

The workflow is: experimental geometry → sample and mask → illumination → exit waves → ideal holograms → corrupted detector images → FTH reconstructions.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
plt.rcParams["figure.constrained_layout.use"] = True


## 1. Define the experimental geometry

The detector size, pixel pitch, sample-to-detector distance, photon energy, and oversampling determine the real-space sample pixel size. The beamstop is part of the detector model and affects only the corrupted measurement—not the ideal hologram.


In [ ]:
detector_shape = (64, 64)
detector_pixel_size = 20e-6       # m
sample_detector_distance = 0.075  # m
detector_center = tuple(np.array(detector_shape) // 2)
oversampling = 2

xray = sim.XRayConfig(
    energy=778.0, photon_flux=5e8, pol="CR",
    coherence_length=(10e-6, 10e-6),
)
xray.setup()

beamstop = sim.BeamstopConfig(
    bs_method="circular", bs_detector_distance=0.010,
    bs_center=detector_center,
    bs_config={
        "radius": 120e-6, "sigma": 8e-6,
        "wire_width": 35e-6, "wire_bend": 15e-6,
        "angle": np.deg2rad(20), "antialias": 3, "seed": 4,
    },
)
detector = sim.DetectorConfig(
    shape=detector_shape, pixel_size=detector_pixel_size,
    sample_to_detector_distance=sample_detector_distance,
    detector_center=detector_center, beamstop_config=beamstop,
    detector_params={
        "readout_noise_average": 20, "readout_noise_sigma": 3,
        "detector_threshold": 5e4, "counts_per_photon": 100,
        "quantum_efficiency": 0.9, "noise_seed": 12,
    },
    measurement_config={"exposure_time": 1.0, "number_frames": 1},
)
detector.setup()
real_space_pixel_size = detector.calc_realspace_resolution(xray.beam_params) / oversampling
sample_shape = [0, oversampling * detector_shape[0], oversampling * detector_shape[1]]
print(f"detector: {detector_shape}, distance: {sample_detector_distance:.3f} m")
print(f"sample grid: {sample_shape[1:]}, pixel: {real_space_pixel_size * 1e9:.2f} nm")


## 2. Define the sample geometry

The recipe defines the material stack. A binary-domain generator supplies the magnetic pattern, and the FTH mask opens one object hole (OH) and two reference holes (RH). All lengths in configuration objects are physical SI values.


In [ ]:
sample = sim.SampleConfig(
    recipe="Au(1000)/SiN(80)/Pt(4)Co(18)Pt(2)",
    sample_shape=sample_shape, real_space_pixel_size=real_space_pixel_size,
    xray_config=xray, sample_name="end-to-end magnetic FTH tutorial",
)
sample.setup()

magnetic = sim.MagneticPatternConfig(
    pattern_type_method="binary_labyrinth_pattern",
    shape=tuple(sample_shape[1:]), real_space_pixel_size=real_space_pixel_size,
    pattern_config={
        "stripe_width": 180e-9, "sigma": 5e-9,
        "H": 128, "W": 128, "n_steps": 60,
        "region": "custom", "use_gpu": False, "seed": 7,
        "k0": 1.05, "eps": 0.8, "target_mean": 0.0,
        "noise_amp": 0.0, "quadratic_coefficient": 0.0,
        "max_hole_area": 9, "saturation_fraction_threshold": 0.01,
    },
)
magnetic.create_pattern()
mz = magnetic.magnetic_pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=np.zeros_like(mz),
    magnetic_pattern_y=np.sqrt(np.clip(1.0 - mz**2, 0.0, 1.0)),
    magnetic_pattern_z=mz,
    nr_repeats=sample.sample_structure.sample_shape[0],
)
sample.assign_magnetic_pattern(magnetization)
print("layers:", sample.sample_structure.layer_names)


In [ ]:
thicknesses = sample.sample_structure.layer_thicknesses
membrane_index = sample.sample_structure.layer_names.index("SiN")
aperture = sim.FrontApertureConfig(
    aperture_method="FTH_circular", aperture_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=thicknesses,
    aperture_layer_names=sample.sample_structure.layer_names,
    aperture_config={
        "apertures_type": ["OH", "RH", "RH"],
        "apertures_radius": [700e-9, 55e-9, 40e-9],
        "apertures_center": [(0.0, 0.0), (1450e-9, -1250e-9), (-1400e-9, -1300e-9)],
        "apertures_sigma": [4e-9, 2e-9, 2e-9],
        "apertures_angle": [0.0, 0.0, 0.0],
        "apertures_ellipticity": [1.0, 1.0, 1.0],
        "apertures_roughness": [0.0, 0.0, 0.0],
        "apertures_roughness_modes": [(0, 0), (0, 0), (0, 0)],
        "apertures_seed": [1, 2, 3],
        "apertures_top_radius_factor": [1.0, 1.0, 1.0],
        "aperture_taper_depth": 0.0,
        "thickness_OH": float(np.sum(thicknesses[:membrane_index])),
    },
    use_roi=True,
)
aperture.setup()
aperture_mask = aperture.return_aperture()
sample.assign_aperture_mask(aperture_mask)
sample.sample_structure.calculate_final_dielectric_tensor(use_aperture_roi=True, compact=True)

mask_projection = np.max(aperture_mask, axis=0)
extent_um = np.array([-sample_shape[2]/2, sample_shape[2]/2, sample_shape[1]/2, -sample_shape[1]/2]) * real_space_pixel_size * 1e6
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].imshow(mz, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[0].set_title("magnetic pattern")
axes[1].imshow(mask_projection, cmap="gray", extent=extent_um); axes[1].set_title("OH + RH mask")
axes[2].imshow(mz * mask_projection, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[2].set_title("visible domains")
for ax in axes: ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")


## 3. Define CR and CL illumination

The spatial Gaussian is created once. `update_polarization` switches its Jones vector between the matched circular helicities without rebuilding the scalar field.


In [ ]:
illumination = sim.IlluminationConfig(
    XRayConfig=xray, shape=tuple(sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={
        "center": np.array([0.0, 0.0]), "distance": 0.0,
        "fwhm": 4.0e-6, "alpha_beam": (0.0, 0.0),
    },
)
illumination.setup()
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(np.abs(illumination.illumination.illumination)**2, cmap="magma", extent=extent_um)
ax.set_title("incident Gaussian intensity"); ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")


## 4. Simulate exit waves and detector holograms

For each helicity, Jones propagation produces the complex exit wave and the ideal far-field intensity. `DetectorConfig` then projects that intensity onto the physical detector and applies finite coherence, the beamstop, photon statistics, readout noise, and detector threshold.


In [ ]:
results = {}
for helicity in ("CR", "CL"):
    illumination.update_polarization(helicity)
    propagation = sim.SamplePropagatorConfig(
        SampleConfig=sample, IlluminationConfig=illumination,
        propagator_method="Jones",
        propagator_config={
            "propagate": False, "jones_apply_zero_order_phase": True,
            "dielectric_tensor_use_roi": True,
        },
    )
    propagation.setup()
    detector.assign_propagated_wavefront(propagation)
    detector.detect_hologram()
    results[helicity] = {
        "exit": propagation.return_scalar_wavefield().copy(),
        "ideal": detector.return_ideal_hologram().copy(),
        "detected": detector.return_detected_hologram().copy(),
    }
print("simulated:", list(results))


## 5. Inspect the CR and CL exit waves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for row, helicity in enumerate(("CR", "CL")):
    field = results[helicity]["exit"]
    amp = axes[row, 0].imshow(np.abs(field), cmap="magma", extent=extent_um)
    phase = axes[row, 1].imshow(np.angle(field), cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent_um)
    axes[row, 0].set_title(f"{helicity} exit amplitude")
    axes[row, 1].set_title(f"{helicity} exit phase")
    fig.colorbar(amp, ax=axes[row, 0], shrink=0.75); fig.colorbar(phase, ax=axes[row, 1], shrink=0.75)


## 6. Ideal and corrupted holograms: sum and difference

The sum `CR + CL` emphasizes nonmagnetic/charge scattering. The difference `CR - CL` isolates helicity-dependent magnetic contrast. The same algebra is applied to the ideal and corrupted detector images.


In [ ]:
holograms = {}
for source in ("ideal", "detected"):
    cr, cl = results["CR"][source], results["CL"][source]
    holograms[source] = {"CR": cr, "CL": cl, "sum": cr + cl, "difference": cr - cl}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("CR", "CL", "sum", "difference")):
        data = holograms[source][channel]
        if channel == "difference":
            limit = max(np.max(np.abs(data)), np.finfo(float).eps)
            image = axes[row, col].imshow(data, cmap="RdBu_r", vmin=-limit, vmax=limit)
        else:
            floor = max(np.max(data) * 1e-8, np.finfo(float).tiny)
            image = axes[row, col].imshow(np.log10(np.maximum(data, floor)), cmap="magma")
        axes[row, col].set_title(f"{source} {channel}"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.72)


## 7. Reconstruct sums and differences

An FTH reconstruction is the centered Fourier transform of the detector hologram. Reconstructions contain displaced object images around each reference-hole correlation peak. Use the same display scale within each channel to compare ideal and corrupted data fairly.


In [ ]:
def fth_reconstruct(hologram):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))

reconstructions = {
    source: {channel: fth_reconstruct(holograms[source][channel]) for channel in ("sum", "difference")}
    for source in ("ideal", "detected")
}
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("sum", "difference")):
        magnitude = np.abs(reconstructions[source][channel])
        vmax = np.percentile(magnitude, 99.5)
        image = axes[row, col].imshow(magnitude, cmap="inferno", vmin=0, vmax=vmax)
        axes[row, col].set_title(f"{source} |FTH({channel})|"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)


## Interpretation

- CR and CL share the structural scattering but interact oppositely with out-of-plane magnetization.
- Their sum is dominated by charge/structural contrast; their difference emphasizes magnetic circular contrast.
- The beamstop, finite photon statistics, detector response, and readout noise make the detected holograms more realistic and propagate into their reconstructions.
- Keep matched acquisition conditions for CR and CL; otherwise the difference also contains exposure or normalization mismatch.
